## Third model - Using subword level dictionary & special character alignment

In [ ]:
# ============================================================
# Complete Implementation of:
# "Post-ASR Correction for Low-Resource Persian Language"
# WITH SUBWORD TOKENIZATION (SentencePiece BPE)
# ============================================================

# Install required libraries
!pip install -q torch torchaudio transformers datasets pandas openpyxl numpy scikit-learn python-Levenshtein matplotlib seaborn sentencepiece

import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random 
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from collections import Counter
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import sentencepiece as spm
warnings.filterwarnings("ignore")

# ============================================================
# Configuration
# ============================================================
class Config:
    TRAIN_FILE = "/kaggle/input/datasets/zahratavakolii/asr-persian-data/asr_cleaned_train_norm.csv"
    VAL_FILE = "/kaggle/input/datasets/zahratavakolii/asr-persian-data/asr_cleaned_validation_norm.csv"
    TEST_FILE = "/kaggle/input/datasets/zahratavakolii/asr-persian-data/asr_cleaned_test_norm.csv"
    
    OUTPUT_DIR = "/kaggle/working/"
    BEST_MODEL_PATH = "/kaggle/working/best_model.pt"
    CHECKPOINT_PATH = "/kaggle/working/checkpoint.pt"
    TRAIN_HISTORY = "/kaggle/working/training_history.csv"
    DETAILED_RESULTS = "/kaggle/working/detailed_test_results.csv"
    TEST_PREDICTIONS = "/kaggle/working/test_predictions.csv"
    GATE_VISUALIZATION_PATH = "/kaggle/working/gate_visualizations/"
    
    # Subword tokenization settings
    SUBWORD_MODEL_PATH = "/kaggle/working/persian_subword.model"
    SUBWORD_VOCAB_SIZE = 1280
    
    EMBEDDING_DIM = 128
    ENCODER_HIDDEN_SIZE = 256
    DECODER_HIDDEN_SIZE = 512
    ENCODER_NUM_LAYERS = 3
    DECODER_NUM_LAYERS = 3
    DROPOUT = 0.7
    
    # Teacher Forcing with Sigmoid Decay
    TEACHER_FORCING_MAX = 1
    TEACHER_FORCING_MIN = 0.15
    TEACHER_FORCING_DECAY = 0.015
    
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 200
    CLIP_GRAD = 0.5
    CHECKPOINT_INTERVAL = 10
    PATIENCE = 20
    EARLY_STOPPING_METRIC = 'cer'
    
    SENSORY_DROPOUT_PROB = 0.3
    SENSORY_DROPOUT_MODE = 'partial'
    
    PAD_TOKEN = '<pad>'
    SOS_TOKEN = '<sos>'
    EOS_TOKEN = '<eos>'
    GAP_TOKEN = '∅'
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    VOCAB = None

# ============================================================
# Teacher Forcing with Sigmoid Decay
# ============================================================
def get_teacher_forcing(epoch, max_tf=0.7, min_tf=0.1, decay_rate=0.02):
    """
    Sigmoid decay for teacher forcing rate.
    - Smooth decrease: slow at beginning and end, faster in middle
    - Never reaches 0 to maintain stability
    """
    return min_tf + (max_tf - min_tf) * np.exp(-decay_rate * epoch)

# ============================================================
# Train SentencePiece from DataFrames
# ============================================================
def train_sentencepiece_from_dataframes(train_df, val_df, test_df, model_path, vocab_size=2000):
    """
    Train SentencePiece model using ground_truth from all dataframes
    """
    all_texts = []
    
    # Collect from all dataframes
    for df in [train_df, val_df, test_df]:
        if 'ground_truth' in df.columns:
            all_texts.extend(df['ground_truth'].dropna().tolist())
    
    print(f"Total texts for training subword model: {len(all_texts)}")
    
    # Filter empty texts
    all_texts = [t for t in all_texts if t and isinstance(t, str) and len(t.strip()) > 0]
    
    with open('train_texts.txt', 'w', encoding='utf-8') as f:
        for text in all_texts:
            f.write(text.strip() + '\n')
    
    # Train the model
    spm.SentencePieceTrainer.train(
        input='train_texts.txt',
        model_prefix=model_path.replace('.model', ''),
        vocab_size=vocab_size,
        character_coverage=0.9995,
        model_type='bpe',
        pad_id=0,
        bos_id=1,
        eos_id=2,
        unk_id=3,
        pad_piece='<pad>',
        bos_piece='<sos>',
        eos_piece='<eos>',
        unk_piece='<unk>',
        split_digits=True,
        byte_fallback=True,
        user_defined_symbols=['∅'],
    )
    
    print(f"SentencePiece model trained with vocab size: {vocab_size}")
    return model_path

# ============================================================
# Print Subword Vocabulary
# ============================================================
def print_subword_vocab(vocab, num_samples=100):
    """
    Print sample subword tokens from the vocabulary
    """
    print("\n" + "="*60)
    print(f"SUBWORD VOCABULARY (showing {num_samples} samples)")
    print("="*60)
    
    for i in range(min(num_samples, len(vocab))):
        piece = vocab.idx2char[i]
        print(f"{i:4d}: {piece}")
    
    print("="*60)
    
    special_tokens = ['<pad>', '<sos>', '<eos>', '<unk>', '∅']
    special_count = sum(1 for i in range(len(vocab)) if vocab.idx2char[i] in special_tokens)
    
    print(f"Total vocab size: {len(vocab)}")
    print(f"Special tokens: {special_count}")
    print(f"Regular subwords: {len(vocab) - special_count}")
    print("="*60)

# ============================================================
# Subword Vocabulary Builder
# ============================================================
class SubwordVocab:
    def __init__(self, sp_model_path):
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(sp_model_path)
        self.pad_idx = 0
        self.sos_idx = 1
        self.eos_idx = 2
        self.unk_idx = 3
        self.vocab_size = len(self.sp)
        
        # For compatibility with existing code
        self.char2idx = {self.sp.id_to_piece(i): i for i in range(self.vocab_size)}
        self.idx2char = {i: self.sp.id_to_piece(i) for i in range(self.vocab_size)}
        
    def encode(self, text, add_sos=True, add_eos=True):
        """Convert text to subword indices"""
        indices = self.sp.encode(text, out_type=int)
        if add_sos:
            indices = [self.sos_idx] + indices
        if add_eos:
            indices = indices + [self.eos_idx]
        return torch.tensor(indices, dtype=torch.long)
    
    def decode(self, indices, ignore_special=True):
        """Convert subword indices back to text"""
        if ignore_special:
            indices = [idx for idx in indices if idx not in [self.pad_idx, self.sos_idx, self.eos_idx, self.unk_idx]]
        return self.sp.decode(indices)
    
    def __len__(self):
        return self.vocab_size

# ============================================================
# Levenshtein Alignment
# ============================================================
def levenshtein_align(s1, s2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n+1, m+1), dtype=int)
    
    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j
    
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )
    
    align1, align2 = [], []
    i, j = n, m
    
    while i > 0 or j > 0:
        if i == 0:
            align1.append(Config.GAP_TOKEN)
            align2.append(s2[j-1])
            j -= 1
        elif j == 0:
            align1.append(s1[i-1])
            align2.append(Config.GAP_TOKEN)
            i -= 1
        else:
            cost = 0 if s1[i-1] == s2[j-1] else 1
            if dp[i][j] == dp[i-1][j-1] + cost:
                align1.append(s1[i-1])
                align2.append(s2[j-1])
                i -= 1
                j -= 1
            elif dp[i][j] == dp[i-1][j] + 1:
                align1.append(s1[i-1])
                align2.append(Config.GAP_TOKEN)
                i -= 1
            else:
                align1.append(Config.GAP_TOKEN)
                align2.append(s2[j-1])
                j -= 1
    
    return ''.join(reversed(align1)), ''.join(reversed(align2))

# ============================================================
# Dataset with Subword Tokenization
# ============================================================
class ASRCorrectionDataset(Dataset):
    def __init__(self, df, vocab, split_name):
        self.df = df[df['split'] == split_name].reset_index(drop=True)
        self.vocab = vocab
        self.split_name = split_name
        self.encoded_data = []
        
        for _, row in self.df.iterrows():
            aligned1, aligned2 = levenshtein_align(row['whisper'], row['mms'])
            
            src1 = vocab.encode(aligned1, add_sos=False, add_eos=False)
            src2 = vocab.encode(aligned2, add_sos=False, add_eos=False)
            tgt = vocab.encode(row['ground_truth'], add_sos=True, add_eos=True)
            
            if tgt[-1] != vocab.eos_idx:
                tgt = torch.cat([tgt, torch.tensor([vocab.eos_idx])])
            
            self.encoded_data.append({
                'src1': src1,
                'src2': src2,
                'tgt': tgt,
                'id': row['id'],
                'ground_truth': row['ground_truth']
            })
        
        print(f"Loaded {len(self.encoded_data)} samples from {split_name}")
    
    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        item = self.encoded_data[idx]
        
        if random.random() < Config.SENSORY_DROPOUT_PROB:
            if Config.SENSORY_DROPOUT_MODE == 'partial':
                src1 = item['src1'].clone()
                src2 = item['src2'].clone()
                
                if random.random() < 0.5:
                    if len(src1) > 2:
                        start = random.randint(0, len(src1) - 2)
                        end = min(start + random.randint(1, min(3, len(src1) - start)), len(src1))
                        src1[start:end] = self.vocab.pad_idx
                else:
                    if len(src2) > 2:
                        start = random.randint(0, len(src2) - 2)
                        end = min(start + random.randint(1, min(3, len(src2) - start)), len(src2))
                        src2[start:end] = self.vocab.pad_idx
            else:
                if random.random() < 0.5:
                    src1 = torch.full_like(item['src1'], self.vocab.pad_idx)
                    src2 = item['src2']
                else:
                    src1 = item['src1']
                    src2 = torch.full_like(item['src2'], self.vocab.pad_idx)
        else:
            src1 = item['src1']
            src2 = item['src2']
        
        return {
            'src1': src1,
            'src2': src2,
            'tgt': item['tgt'],
            'id': item['id'],
            'ground_truth': item['ground_truth']
        }

# ============================================================
# Collate Function with unified padding length
# ============================================================
def collate_batch(batch):
    src1_list = [item['src1'] for item in batch]
    src2_list = [item['src2'] for item in batch]
    tgt_list = [item['tgt'] for item in batch]
    ids = [item['id'] for item in batch]
    gt_texts = [item['ground_truth'] for item in batch]
    
    pad_idx = Config.VOCAB.pad_idx if Config.VOCAB else 0
    
    # Calculate max length across all sequences
    max_len = 0
    for s1, s2, t in zip(src1_list, src2_list, tgt_list):
        max_len = max(max_len, len(s1), len(s2), len(t))
    
    # Pad all sequences to the same max_len
    padded_src1 = []
    padded_src2 = []
    padded_tgt = []
    
    for i in range(len(src1_list)):
        # Pad src1
        if len(src1_list[i]) < max_len:
            pad = torch.full((max_len - len(src1_list[i]),), pad_idx, dtype=torch.long)
            padded_src1.append(torch.cat([src1_list[i], pad]))
        else:
            padded_src1.append(src1_list[i])
        
        # Pad src2
        if len(src2_list[i]) < max_len:
            pad = torch.full((max_len - len(src2_list[i]),), pad_idx, dtype=torch.long)
            padded_src2.append(torch.cat([src2_list[i], pad]))
        else:
            padded_src2.append(src2_list[i])
        
        # Pad tgt
        if len(tgt_list[i]) < max_len:
            pad = torch.full((max_len - len(tgt_list[i]),), pad_idx, dtype=torch.long)
            padded_tgt.append(torch.cat([tgt_list[i], pad]))
        else:
            padded_tgt.append(tgt_list[i])
    
    # Convert to tensors
    padded_src1 = torch.stack(padded_src1)
    padded_src2 = torch.stack(padded_src2)
    padded_tgt = torch.stack(padded_tgt)
    
    src1_lens = torch.tensor([len(s) for s in src1_list], dtype=torch.long)
    src2_lens = torch.tensor([len(s) for s in src2_list], dtype=torch.long)
    tgt_lens = torch.tensor([len(t) for t in tgt_list], dtype=torch.long)
    
    return {
        'src1': padded_src1,
        'src2': padded_src2,
        'tgt': padded_tgt,
        'src1_lens': src1_lens,
        'src2_lens': src2_lens,
        'tgt_lens': tgt_lens,
        'ids': ids,
        'gt_texts': gt_texts
    }

# ============================================================
# Gated Fusion
# ============================================================
class GatedFusion(nn.Module):
    def __init__(self, hidden_size, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)
        
        self.proj1 = nn.Linear(hidden_size * 2, hidden_size)
        self.proj2 = nn.Linear(hidden_size * 2, hidden_size)
        self.gate12 = nn.Linear(hidden_size * 2, hidden_size)
        self.gate21 = nn.Linear(hidden_size * 2, hidden_size)
        
        self.layer_norm1 = nn.LayerNorm(hidden_size)
        self.layer_norm2 = nn.LayerNorm(hidden_size)
        
        self.last_g12 = None
        self.last_g21 = None
        
    def forward(self, h_t):
        v1 = self.dropout(self.proj1(h_t))
        v2 = self.dropout(self.proj2(h_t))
        
        v1 = self.layer_norm1(v1)
        v2 = self.layer_norm2(v2)
        
        g12 = torch.sigmoid(self.gate12(torch.cat([v1, v2], dim=-1)))
        g21 = torch.sigmoid(self.gate21(torch.cat([v2, v1], dim=-1)))
        
        self.last_g12 = g12.detach()
        self.last_g21 = g21.detach()
        
        h1_hat = (1 - g12) * v1 + g12 * v2
        h2_hat = g21 * v1 + (1 - g21) * v2
        
        h_hat = torch.cat([h1_hat, h2_hat], dim=-1)
        return h_hat

# ============================================================
# Bahdanau Attention
# ============================================================
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        self.W_a = nn.Linear(hidden_size, hidden_size)
        self.U_a = nn.Linear(hidden_size, hidden_size)
        self.V_a = nn.Linear(hidden_size, 1)
    
    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        seq_len = encoder_outputs.size(1)
        decoder_hidden = decoder_hidden.unsqueeze(1).repeat(1, seq_len, 1)
        
        energy = torch.tanh(self.W_a(encoder_outputs) + self.U_a(decoder_hidden))
        attention = self.V_a(energy).squeeze(-1)
        
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        
        attention_weights = torch.softmax(attention, dim=1)
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, attention_weights

# ============================================================
# Multi-View Seq2Seq Model
# ============================================================
class MultiViewGatedSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embedding_dim, enc_hidden_size, dec_hidden_size,
                 enc_num_layers, dec_num_layers, dropout):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.enc_hidden_size = enc_hidden_size
        self.dec_hidden_size = dec_hidden_size
        self.enc_num_layers = enc_num_layers
        self.dec_num_layers = dec_num_layers
        self.dropout = dropout
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.encoder = nn.LSTM(
            input_size=embedding_dim * 2,
            hidden_size=enc_hidden_size,
            num_layers=enc_num_layers,
            bidirectional=True,
            dropout=dropout if enc_num_layers > 1 else 0,
            batch_first=True
        )
        
        self.gated_fusion = GatedFusion(enc_hidden_size)
        self.attention = BahdanauAttention(dec_hidden_size)
        
        self.hidden_projection = nn.Linear(enc_hidden_size, dec_hidden_size)
        
        self.decoder = nn.LSTM(
            input_size=dec_hidden_size + embedding_dim,
            hidden_size=dec_hidden_size,
            num_layers=dec_num_layers,
            dropout=dropout if dec_num_layers > 1 else 0,
            batch_first=True
        )
        
        self.fc = nn.Linear(dec_hidden_size * 2, vocab_size)
        self.dropout_layer = nn.Dropout(dropout)
        
    def forward(self, src1, src2, src1_lens, src2_lens, tgt=None, teacher_forcing_ratio=0.5):
        batch_size = src1.size(0)
        device = src1.device

        emb1 = self.embedding(src1)
        emb2 = self.embedding(src2)
        concat_emb = torch.cat([emb1, emb2], dim=-1)

        sorted_lens, sorted_idx = src1_lens.sort(descending=True)
        sorted_concat_emb = concat_emb[sorted_idx]

        packed_emb = pack_padded_sequence(
            sorted_concat_emb,
            sorted_lens.cpu(),
            batch_first=True,
            enforce_sorted=True
        )
        packed_output, (hidden, cell) = self.encoder(packed_emb)
        enc_outputs, _ = pad_packed_sequence(packed_output, batch_first=True)

        _, unsorted_idx = sorted_idx.sort()
        enc_outputs = enc_outputs[unsorted_idx]
        hidden = hidden[:, unsorted_idx]
        cell = cell[:, unsorted_idx]

        enc_outputs = self.gated_fusion(enc_outputs)
        enc_outputs_proj = enc_outputs

        hidden = hidden.view(self.enc_num_layers, 2, batch_size, self.enc_hidden_size)
        hidden = hidden.mean(dim=1)
        cell = cell.view(self.enc_num_layers, 2, batch_size, self.enc_hidden_size)
        cell = cell.mean(dim=1)

        hidden = self.hidden_projection(hidden.permute(1, 0, 2).contiguous()).permute(1, 0, 2).contiguous()
        cell = self.hidden_projection(cell.permute(1, 0, 2).contiguous()).permute(1, 0, 2).contiguous()

        sos_idx = Config.VOCAB.sos_idx
        eos_idx = Config.VOCAB.eos_idx

        if tgt is not None:
            max_len = tgt.size(1) - 1
        else:
            max_len = 100
        
        max_len = max(1, max_len)
        max_len = min(max_len, 150)

        decoder_input = torch.full((batch_size, 1), sos_idx, dtype=torch.long, device=device)
        decoder_hidden = (hidden, cell)
        
        done = torch.zeros(batch_size, dtype=torch.bool, device=device)
        outputs = torch.zeros(batch_size, max_len, self.vocab_size, device=device)

        for t in range(max_len):
            decoder_emb = self.embedding(decoder_input)
            context, _ = self.attention(decoder_hidden[0][-1], enc_outputs_proj)
            context = context.unsqueeze(1)

            decoder_input_combined = torch.cat([context, decoder_emb], dim=-1)
            output, decoder_hidden = self.decoder(decoder_input_combined, decoder_hidden)

            combined = torch.cat([output.squeeze(1), context.squeeze(1)], dim=-1)
            pred = self.fc(self.dropout_layer(combined))
            outputs[:, t, :] = pred

            if tgt is not None and torch.rand(1).item() < teacher_forcing_ratio:
                next_token = tgt[:, t+1:t+2]
                decoder_input = torch.where(
                    done.unsqueeze(1),
                    torch.full_like(next_token, eos_idx),
                    next_token
                )
            else:
                pred_token = pred.argmax(dim=-1).unsqueeze(1)
                newly_done = (pred_token.squeeze(-1) == eos_idx) & ~done
                done = done | newly_done
                
                decoder_input = torch.where(
                    done.unsqueeze(1),
                    torch.full_like(pred_token, eos_idx),
                    pred_token
                )

        return outputs

    def predict(self, src1, src2, src1_lens, src2_lens, max_len=150):
        max_len = max(1, max_len)
        
        batch_size = src1.size(0)
        device = src1.device
        pad_idx = Config.VOCAB.pad_idx if Config.VOCAB else 0
        
        max_seq_len = max(src1.size(1), src2.size(1))
        
        if src1.size(1) < max_seq_len:
            pad = torch.full((batch_size, max_seq_len - src1.size(1)), pad_idx, dtype=torch.long, device=device)
            src1 = torch.cat([src1, pad], dim=1)
        
        if src2.size(1) < max_seq_len:
            pad = torch.full((batch_size, max_seq_len - src2.size(1)), pad_idx, dtype=torch.long, device=device)
            src2 = torch.cat([src2, pad], dim=1)
        
        # Update src1_lens and src2_lens
        src1_lens = torch.full((batch_size,), max_seq_len, device=device)
        src2_lens = torch.full((batch_size,), max_seq_len, device=device)
        # ===========================================================
        
        emb1 = self.embedding(src1)
        emb2 = self.embedding(src2)
        concat_emb = torch.cat([emb1, emb2], dim=-1)
        
        sorted_lens, sorted_idx = src1_lens.sort(descending=True)
        sorted_concat_emb = concat_emb[sorted_idx]
        packed_emb = pack_padded_sequence(
            sorted_concat_emb,
            sorted_lens.cpu(),
            batch_first=True,
            enforce_sorted=True
        )
        packed_output, (hidden, cell) = self.encoder(packed_emb)
        enc_outputs, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        _, unsorted_idx = sorted_idx.sort()
        enc_outputs = enc_outputs[unsorted_idx]
        hidden = hidden[:, unsorted_idx]
        cell = cell[:, unsorted_idx]
        
        enc_outputs = self.gated_fusion(enc_outputs)
        enc_outputs_proj = enc_outputs
        
        hidden = hidden.view(self.enc_num_layers, 2, batch_size, self.enc_hidden_size)
        hidden = hidden.mean(dim=1)
        cell = cell.view(self.enc_num_layers, 2, batch_size, self.enc_hidden_size)
        cell = cell.mean(dim=1)
        
        hidden = self.hidden_projection(hidden.permute(1, 0, 2).contiguous()).permute(1, 0, 2).contiguous()
        cell = self.hidden_projection(cell.permute(1, 0, 2).contiguous()).permute(1, 0, 2).contiguous()
        
        sos_idx = Config.VOCAB.sos_idx
        eos_idx = Config.VOCAB.eos_idx
        
        decoder_input = torch.full((batch_size, 1), sos_idx, dtype=torch.long, device=device)
        decoder_hidden = (hidden, cell)
        
        done = torch.zeros(batch_size, dtype=torch.bool, device=device)
        predictions = torch.full((batch_size, max_len), pad_idx, dtype=torch.long, device=device)
        
        for step in range(max_len):
            decoder_emb = self.embedding(decoder_input)
            context, _ = self.attention(decoder_hidden[0][-1], enc_outputs_proj)
            context = context.unsqueeze(1)
            
            decoder_input_combined = torch.cat([context, decoder_emb], dim=-1)
            output, decoder_hidden = self.decoder(decoder_input_combined, decoder_hidden)
            
            combined = torch.cat([output.squeeze(1), context.squeeze(1)], dim=-1)
            pred = self.fc(self.dropout_layer(combined))
            
            next_token = pred.argmax(dim=-1)
            predictions[:, step] = next_token
            
            newly_done = (next_token == eos_idx) & ~done
            done = done | newly_done
            
            decoder_input = torch.where(
                done.unsqueeze(1),
                torch.full_like(next_token.unsqueeze(1), eos_idx),
                next_token.unsqueeze(1)
            )
        
        for i in range(batch_size):
            seq = predictions[i]
            if eos_idx in seq:
                eos_pos = (seq == eos_idx).nonzero(as_tuple=True)[0][0].item()
                predictions[i, eos_pos+1:] = pad_idx
        
        return predictions

# ============================================================
# Metrics
# ============================================================
def calculate_cer(reference, hypothesis):
    if len(reference) == 0:
        return 1.0 if len(hypothesis) > 0 else 0.0
    n, m = len(reference), len(hypothesis)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if reference[i-1] == hypothesis[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[n][m] / len(reference)

def calculate_wer(reference, hypothesis):
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if len(ref_words) == 0:
        return 1.0 if len(hyp_words) > 0 else 0.0
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref_words[i-1] == hyp_words[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[n][m] / len(ref_words)

def compute_metrics(predictions, ground_truths):
    total_cer = 0
    total_wer = 0
    count = len(predictions)
    for pred, gt in zip(predictions, ground_truths):
        total_cer += calculate_cer(gt, pred)
        total_wer += calculate_wer(gt, pred)
    return total_cer / count, total_wer / count

# ============================================================
# Training Functions
# ============================================================
def train_epoch(model, dataloader, optimizer, criterion, teacher_forcing_ratio, clip_grad=0.5):
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch in dataloader:
        src1 = batch['src1'].to(Config.DEVICE)
        src2 = batch['src2'].to(Config.DEVICE)
        tgt = batch['tgt'].to(Config.DEVICE)
        src1_lens = batch['src1_lens']
        src2_lens = batch['src2_lens']
        
        optimizer.zero_grad()
        outputs = model(src1, src2, src1_lens, src2_lens, tgt, teacher_forcing_ratio)
        
        pad_idx = Config.VOCAB.pad_idx if Config.VOCAB else 0
        
        tgt_lens = batch['tgt_lens']
        loss = 0
        batch_size = tgt.size(0)
        max_len = outputs.size(1)
        
        for i in range(batch_size):
            seq_len = min(tgt_lens[i].item(), max_len)
            if seq_len > 0:
                seq_loss = criterion(
                    outputs[i, :seq_len-1, :],
                    tgt[i, 1:seq_len]
                )
                loss += seq_loss
        
        if batch_size > 0:
            loss = loss / batch_size
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else 0.0

def evaluate(model, dataloader, criterion, teacher_forcing_ratio=0.0, return_raw_outputs=False):
    model.eval()
    total_loss = 0
    num_batches = 0
    all_predictions = []
    all_ground_truths = []
    all_ids = []
    
    with torch.no_grad():
        for batch in dataloader:
            src1 = batch['src1'].to(Config.DEVICE)
            src2 = batch['src2'].to(Config.DEVICE)
            tgt = batch['tgt'].to(Config.DEVICE)
            src1_lens = batch['src1_lens']
            src2_lens = batch['src2_lens']
            
            outputs = model(src1, src2, src1_lens, src2_lens, tgt, teacher_forcing_ratio)
            
            pad_idx = Config.VOCAB.pad_idx if Config.VOCAB else 0
            
            tgt_lens = batch['tgt_lens']
            loss = 0
            batch_size = tgt.size(0)
            max_len = outputs.size(1)
            
            for i in range(batch_size):
                seq_len = min(tgt_lens[i].item(), max_len)
                if seq_len > 0:
                    seq_loss = criterion(
                        outputs[i, :seq_len-1, :],
                        tgt[i, 1:seq_len]
                    )
                    loss += seq_loss
            
            if batch_size > 0:
                loss = loss / batch_size
            
            total_loss += loss.item()
            num_batches += 1
            
            preds = outputs.argmax(dim=-1)
            
            for i in range(len(tgt)):
                gt_tokens = tgt[i][1:].tolist()
                eos_pos_gt = gt_tokens.index(Config.VOCAB.eos_idx) if Config.VOCAB.eos_idx in gt_tokens else len(gt_tokens)
                gt_tokens = gt_tokens[:eos_pos_gt]
                gt_text = Config.VOCAB.decode(gt_tokens, ignore_special=True)
                all_ground_truths.append(gt_text)
                
                pred_tokens = preds[i].tolist()
                eos_pos_pred = pred_tokens.index(Config.VOCAB.eos_idx) if Config.VOCAB.eos_idx in pred_tokens else len(pred_tokens)
                pred_tokens = pred_tokens[:eos_pos_pred]
                pred_text = Config.VOCAB.decode(pred_tokens, ignore_special=True)
                all_predictions.append(pred_text)
            
            all_ids.extend(batch['ids'])
    
    if return_raw_outputs:
        df = dataloader.dataset.df
        id_to_whisper = dict(zip(df['id'], df['whisper']))
        id_to_mms = dict(zip(df['id'], df['mms']))
        
        all_whisper = [id_to_whisper.get(sid, '') for sid in all_ids]
        all_mms = [id_to_mms.get(sid, '') for sid in all_ids]
        
        return total_loss / num_batches if num_batches > 0 else 0.0, all_predictions, all_ground_truths, all_ids, all_whisper, all_mms
    
    return total_loss / num_batches if num_batches > 0 else 0.0, all_predictions, all_ground_truths, all_ids

# ============================================================
# Display Function
# ============================================================
def display_fixed_and_random_samples(model, dataset, vocab, epoch, fixed_idx=0):
    model.eval()
    random_idx = random.randint(0, len(dataset) - 1)
    samples = [('Fixed (Index 0)', fixed_idx), ('Random', random_idx)]
    
    print("\n" + "="*70)
    print(f"SAMPLE OUTPUTS - Epoch {epoch}")
    
    with torch.no_grad():
        for name, idx in samples:
            item = dataset[idx]
            src1 = item['src1'].unsqueeze(0).to(Config.DEVICE)
            src2 = item['src2'].unsqueeze(0).to(Config.DEVICE)
            length = torch.tensor([len(item['src1'])])
            
            pred_tokens = model.predict(src1, src2, length, length, max_len=100)
            pred_text = vocab.decode(pred_tokens[0].tolist(), ignore_special=True)
            gt_text = vocab.decode(item['tgt'].tolist(), ignore_special=True)
            
            row = dataset.df.iloc[idx]
            whisper_text = row['whisper']
            mms_text = row['mms']
            
            cer = calculate_cer(gt_text, pred_text)
            wer = calculate_wer(gt_text, pred_text)
            
            print(f"\n{name} (ID: {item['id']})")
            print(f"  Whisper:     {whisper_text}")
            print(f"  MMS:         {mms_text}")
            print(f"  Ground Truth:{gt_text}")
            print(f"  Model Output:{pred_text}")
            print(f"  CER: {cer:.4f} | WER: {wer:.4f}")
            print("  " + "-"*50)
    
    print("="*70)

# ============================================================
# Gate Analysis
# ============================================================
def analyze_gates(model, dataloader, vocab, num_samples=3):
    model.eval()
    gate_stats = {'g12_values': [], 'g21_values': [], 'positions': []}
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= num_samples:
                break
            src1 = batch['src1'].to(Config.DEVICE)
            src2 = batch['src2'].to(Config.DEVICE)
            tgt = batch['tgt'].to(Config.DEVICE)
            src1_lens = batch['src1_lens']
            src2_lens = batch['src2_lens']
            
            _ = model(src1, src2, src1_lens, src2_lens, tgt, teacher_forcing_ratio=0.0)
            g12 = model.gated_fusion.last_g12.mean(dim=-1)
            g21 = model.gated_fusion.last_g21.mean(dim=-1)
            
            for i in range(min(g12.size(0), 3)):
                seq_len = g12.size(1)
                gate_stats['positions'].extend(range(seq_len))
                gate_stats['g12_values'].extend(g12[i, :seq_len].cpu().tolist())
                gate_stats['g21_values'].extend(g21[i, :seq_len].cpu().tolist())
    return gate_stats

def visualize_gates(gate_stats, save_path=None):
    if not gate_stats['g12_values']:
        print("No gate data")
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(gate_stats['g12_values'], bins=30, alpha=0.5, label='g12 (Whisper)')
    axes[0].hist(gate_stats['g21_values'], bins=30, alpha=0.5, label='g21 (MMS)')
    axes[0].set_xlabel('Gate Value')
    axes[0].legend()
    
    positions = np.array(gate_stats['positions'])
    g12_arr = np.array(gate_stats['g12_values'])
    g21_arr = np.array(gate_stats['g21_values'])
    unique_pos = sorted(set(positions))
    g12_avg = [np.mean(g12_arr[positions == p]) for p in unique_pos]
    g21_avg = [np.mean(g21_arr[positions == p]) for p in unique_pos]
    axes[1].plot(unique_pos, g12_avg, 'b-', label='g12')
    axes[1].plot(unique_pos, g21_avg, 'r-', label='g21')
    axes[1].set_xlabel('Position')
    axes[1].legend()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.show()

# ============================================================
# Main Training Loop
# ============================================================
def main():
    print("="*50)
    print("ASR CORRECTION WITH SUBWORD TOKENIZATION")
    print("="*50)
    print(f"Device: {Config.DEVICE}")
    print(f"Subword Vocab Size: {Config.SUBWORD_VOCAB_SIZE}")
    print(f"Encoder Hidden Size: {Config.ENCODER_HIDDEN_SIZE}")
    print(f"Decoder Hidden Size: {Config.DECODER_HIDDEN_SIZE}")
    print(f"Teacher Forcing: Sigmoid Decay (max={Config.TEACHER_FORCING_MAX}, min={Config.TEACHER_FORCING_MIN})")
    print(f"Sensory Dropout: {Config.SENSORY_DROPOUT_PROB} (mode: {Config.SENSORY_DROPOUT_MODE})")
    
    for file_path in [Config.TRAIN_FILE, Config.VAL_FILE, Config.TEST_FILE]:
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: File not found: {file_path}")
    
    print("\nLoading datasets...")
    train_df = pd.read_csv(Config.TRAIN_FILE)
    val_df = pd.read_csv(Config.VAL_FILE)
    test_df = pd.read_csv(Config.TEST_FILE)
    print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")
    
    # ============================================================
    # Train SentencePiece Model
    # ============================================================
    print("\nTraining SentencePiece model from ground_truth columns...")
    Config.SUBWORD_MODEL_PATH = train_sentencepiece_from_dataframes(
        train_df, val_df, test_df,
        Config.SUBWORD_MODEL_PATH,
        Config.SUBWORD_VOCAB_SIZE
    )
    
    print("\nBuilding vocabulary...")
    Config.VOCAB = SubwordVocab(Config.SUBWORD_MODEL_PATH)
    print(f"Vocab size: {len(Config.VOCAB)}")
    
    # Print subword vocabulary
    print_subword_vocab(Config.VOCAB, num_samples=Config.SUBWORD_VOCAB_SIZE)
    
    print("\nCreating datasets...")
    train_dataset = ASRCorrectionDataset(train_df, Config.VOCAB, 'train')
    val_dataset = ASRCorrectionDataset(val_df, Config.VOCAB, 'validation')
    test_dataset = ASRCorrectionDataset(test_df, Config.VOCAB, 'test')
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=0)
    
    print("\nInitializing model...")
    model = MultiViewGatedSeq2Seq(
        vocab_size=len(Config.VOCAB),
        embedding_dim=Config.EMBEDDING_DIM,
        enc_hidden_size=Config.ENCODER_HIDDEN_SIZE,
        dec_hidden_size=Config.DECODER_HIDDEN_SIZE,
        enc_num_layers=Config.ENCODER_NUM_LAYERS,
        dec_num_layers=Config.DECODER_NUM_LAYERS,
        dropout=Config.DROPOUT
    ).to(Config.DEVICE)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-5)
    pad_idx = Config.VOCAB.pad_idx if Config.VOCAB else 0
    class_weights = torch.ones(len(Config.VOCAB))
    class_weights[Config.VOCAB.eos_idx] = 10.0
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(Config.DEVICE), ignore_index=pad_idx)
    
    print("\nStarting training...\n")
    best_val_cer = float('inf')
    patience_counter = 0
    history = []
    os.makedirs(Config.GATE_VISUALIZATION_PATH, exist_ok=True)
    
    for epoch in range(1, Config.NUM_EPOCHS + 1):
        # Sigmoid decay for teacher forcing
        current_tf = get_teacher_forcing(
            epoch, 
            Config.TEACHER_FORCING_MAX, 
            Config.TEACHER_FORCING_MIN, 
            Config.TEACHER_FORCING_DECAY
        )
        
        train_loss = train_epoch(model, train_loader, optimizer, criterion, current_tf, Config.CLIP_GRAD)
        val_loss, val_preds, val_gts, val_ids = evaluate(model, val_loader, criterion, teacher_forcing_ratio=0.0)
        val_cer, val_wer = compute_metrics(val_preds, val_gts)
        
        print(f"Epoch {epoch:3d}/{Config.NUM_EPOCHS} | TF: {current_tf:.3f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | CER: {val_cer:.4f} | WER: {val_wer:.4f}")
        
        display_fixed_and_random_samples(model, val_dataset, Config.VOCAB, epoch, fixed_idx=0)
        
        if epoch % 10 == 0:
            gate_stats = analyze_gates(model, val_loader, Config.VOCAB, num_samples=3)
            visualize_gates(gate_stats, save_path=os.path.join(Config.GATE_VISUALIZATION_PATH, f"gates_epoch_{epoch}.png"))
        
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_cer': val_cer,
            'val_wer': val_wer,
            'teacher_forcing': current_tf
        })
        
        if val_cer < best_val_cer:
            best_val_cer = val_cer
            patience_counter = 0
            torch.save(model.state_dict(), Config.BEST_MODEL_PATH)
            print(f"  ✅ New best model saved (CER: {val_cer:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= Config.PATIENCE:
                print(f"⏹️ Early stopping triggered after {epoch} epochs")
                break
        
        if epoch % Config.CHECKPOINT_INTERVAL == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_cer': best_val_cer,
            }, Config.CHECKPOINT_PATH)
    
    history_df = pd.DataFrame(history)
    history_df.to_csv(Config.TRAIN_HISTORY, index=False)
    
    print("\n" + "="*50)
    print("FINAL EVALUATION ON TEST SET")
    print("="*50)
    
    model.load_state_dict(torch.load(Config.BEST_MODEL_PATH, map_location=Config.DEVICE))
    test_loss, test_preds, test_gts, test_ids, test_whisper, test_mms = evaluate(
        model, test_loader, criterion, teacher_forcing_ratio=0.0, return_raw_outputs=True
    )
    test_cer, test_wer = compute_metrics(test_preds, test_gts)
    
    print(f"Test Set Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  CER:  {test_cer:.4f}")
    print(f"  WER:  {test_wer:.4f}")
    
    detailed_results = pd.DataFrame({
        'id': test_ids,
        'ground_truth': test_gts,
        'whisper_input': test_whisper,
        'mms_input': test_mms,
        'model_prediction': test_preds,
        'cer': [calculate_cer(gt, pred) for gt, pred in zip(test_gts, test_preds)],
        'wer': [calculate_wer(gt, pred) for gt, pred in zip(test_gts, test_preds)],
        'is_correct': [gt == pred for gt, pred in zip(test_gts, test_preds)]
    })
    detailed_results.to_csv(Config.DETAILED_RESULTS, index=False)
    print(f"Detailed results saved to: {Config.DETAILED_RESULTS}")
    
    print("\n" + "="*50)
    print("COMPARISON WITH BASELINES")
    print("="*50)
    
    test_df_raw = pd.read_csv(Config.TEST_FILE)
    whisper_cer = 0
    whisper_wer = 0
    mms_cer = 0
    mms_wer = 0
    
    for _, row in test_df_raw.iterrows():
        gt = row['ground_truth']
        whisper_cer += calculate_cer(gt, row['whisper'])
        whisper_wer += calculate_wer(gt, row['whisper'])
        mms_cer += calculate_cer(gt, row['mms'])
        mms_wer += calculate_wer(gt, row['mms'])
    
    n = len(test_df_raw)
    whisper_cer /= n
    whisper_wer /= n
    mms_cer /= n
    mms_wer /= n
    
    print(f"Whisper Baseline - CER: {whisper_cer:.4f}, WER: {whisper_wer:.4f}")
    print(f"MMS Baseline     - CER: {mms_cer:.4f}, WER: {mms_wer:.4f}")
    print(f"Our Model        - CER: {test_cer:.4f}, WER: {test_wer:.4f}")
    
    if whisper_cer > 0:
        cer_improvement = ((whisper_cer - test_cer) / whisper_cer) * 100
        wer_improvement = ((whisper_wer - test_wer) / whisper_wer) * 100
        print(f"\nImprovement over Whisper:")
        print(f"  CER: {cer_improvement:.2f}% reduction")
        print(f"  WER: {wer_improvement:.2f}% reduction")
    
    print("\n" + "="*50)
    print("ALL FILES SAVED IN /kaggle/working/")
    print("="*50)
    print("Training complete!")
    
    return model, history_df, detailed_results

# ============================================================
# Run
# ============================================================
if __name__ == "__main__":
    model, history, detailed_results = main()